In [1]:
import requests
import json
import time
from tqdm import tqdm

In [2]:
with open('rulemaker.json', 'r') as file:
    bodyTemplate = json.load(file)

In [3]:
bodyTemplate

[{'name': 'LoadIndex',
  'parameters': {'path': '/kg-microbe/kgm_train.index', 'partially': False}},
 {'name': 'Mine',
  'parameters': {'thresholds': [{'name': 'MinSupport', 'value': 50},
    {'name': 'MaxRuleLength', 'value': 4}],
   'ruleConsumers': [{'name': 'topK', 'k': 100000, 'allowOverflow': True}],
   'patterns': [{'body': [],
     'head': {'subject': {'name': 'Variable', 'value': 'a'},
      'predicate': {'name': 'Constant', 'value': 'biolink:occurs_in'},
      'object': {'name': 'Constant', 'value': 'medm:693'},
      'graph': {'name': 'Any'}},
     'exact': False,
     'orderless': True}],
   'constraints': [{'name': 'WithoutPredicates',
     'values': ['<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
      'rdfs:label']},
    {'name': 'OnlyLowerCardinalitySideConstants'},
    {'name': 'WithoutDuplicitPredicates'}],
   'parallelism': 0}},
 {'name': 'ComputeSupport', 'parameters': {'name': 'Support', 'min': 50}},
 {'name': 'ExportRules',
  'parameters': {'path': '/kg-micr

In [5]:
with open('support_rulemaker.json', 'r') as file:
    top40 = json.load(file)

In [6]:
top40

{'https://mediadive.dsmz.de/medium/65': 1721,
 'https://mediadive.dsmz.de/medium/514': 1277,
 'https://mediadive.dsmz.de/medium/693': 1157,
 'https://mediadive.dsmz.de/medium/92': 1060,
 'https://mediadive.dsmz.de/medium/830': 1033,
 'https://mediadive.dsmz.de/medium/84': 786,
 'https://mediadive.dsmz.de/medium/J14': 535,
 'https://mediadive.dsmz.de/medium/553': 394,
 'https://mediadive.dsmz.de/medium/104': 386,
 'https://mediadive.dsmz.de/medium/215': 284,
 'https://mediadive.dsmz.de/medium/554': 271,
 'https://mediadive.dsmz.de/medium/11': 260,
 'https://mediadive.dsmz.de/medium/110': 194,
 'https://mediadive.dsmz.de/medium/78': 179,
 'https://mediadive.dsmz.de/medium/645': 178,
 'https://mediadive.dsmz.de/medium/1a': 163,
 'https://mediadive.dsmz.de/medium/545': 158,
 'https://mediadive.dsmz.de/medium/58': 150,
 'https://mediadive.dsmz.de/medium/381': 127,
 'https://mediadive.dsmz.de/medium/339': 119,
 'https://mediadive.dsmz.de/medium/104b': 100,
 'https://mediadive.dsmz.de/medium/

In [4]:
bodyTemplate[2]["parameters"]["min"]

50

In [5]:
bodyTemplate[3]["parameters"]["path"]

'/kg-microbe/kg-microbe_media_rules'

In [7]:
row_id = 0

for row in tqdm(top40):
    row_id+=1
    support = int(top40[row]*0.03)

    bodyTemplate[1]["parameters"]["patterns"][0]["head"]["object"]["value"] = "<"+row+">"
    bodyTemplate[1]["parameters"]["thresholds"][0]["value"] = support
    bodyTemplate[2]["parameters"]["min"] = support
    bodyTemplate[3]["parameters"]["path"] = '/kg-microbe/'+row.split("medium/")[1]+'.ndjson'

    URL = "https://precog.vse.cz/rdfrules/api/task"

    body = bodyTemplate

    r = requests.post(url = URL, json = body)

    data = r.json()

    id = data['id']
    print("Odeslal se pozadavek na nalezeni pravidel k mediu ", row, ", task id je ", id, "support je ", support)
    print(bodyTemplate)

    #get requests
    getURL = URL + '/' +id

    r = requests.get(getURL)

    time.sleep(15)

    while r.status_code == 202:
        r = requests.get(getURL)
        print(row, "(", row_id, "/", len(top40), ")", r.status_code," ... LOGY:", r.json()['logs'])
        time.sleep(15)
    else:
        if r.status_code == 200:
            print("\n Pravidla nalezena k mediu ", row, "(", row_id, "/", len(top40), ")", "\n")
        elif r.status_code != 202 and r.status_code != 200:
            print("\n v mediu ", row, " task ", id, " nastala chyba ", r.status_code, "\n")

Odeslal se pozadavek na nalezeni pravidel k mediu  https://mediadive.dsmz.de/medium/65 , task id je  b8a46a3d-3810-451f-a47e-dfb851917d49 support je  51
[{'name': 'LoadIndex', 'parameters': {'path': '/kg-microbe/kgm_train.index', 'partially': False}}, {'name': 'Mine', 'parameters': {'thresholds': [{'name': 'MinSupport', 'value': 51}, {'name': 'MaxRuleLength', 'value': 4}], 'ruleConsumers': [{'name': 'topK', 'k': 100000, 'allowOverflow': True}], 'patterns': [{'body': [], 'head': {'subject': {'name': 'Variable', 'value': 'a'}, 'predicate': {'name': 'Constant', 'value': 'biolink:occurs_in'}, 'object': {'name': 'Constant', 'value': '<https://mediadive.dsmz.de/medium/65>'}, 'graph': {'name': 'Any'}}, 'exact': False, 'orderless': True}], 'constraints': [{'name': 'WithoutPredicates', 'values': ['<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', 'rdfs:label']}, {'name': 'OnlyLowerCardinalitySideConstants'}, {'name': 'WithoutDuplicitPredicates'}], 'parallelism': 0}}, {'name': 'ComputeSupport'

In [7]:
merged_content = ''
all_rules_file = 'kg-microbe_media_rules.ndjson'
# the API endpoint for files
url = "https://precog.vse.cz/rdfrules/api/workspace/kg-microbe/"

for row in tqdm(top40):
    try:
        filename = row.split("medium/")[1] + '.ndjson'
        response = requests.get(url + filename)  # bad responses (4xx or 5xx)
        print(f"File: {filename}, Status: {response.status_code}, Size: {len(response.text)}")

        merged_content += response.text + "\n"

    except requests.exceptions.RequestException as e:
        print(f'Error retrieving file: {e}')
        print('Aborting merge process.')

try:
    with open(all_rules_file, 'w', encoding='utf-8') as f:
        f.write(merged_content)

    print(f'\nAll .ndjson files successfully merged and saved to {all_rules_file}')

except IOError as e:
    print(f'Error saving merged file: {e}')


All .ndjson files successfully merged and saved to kg-microbe_media_rules.ndjson
